# Phase 3 — Fine-tune LayoutLMv3 on SROIE

This Colab notebook fine-tunes `microsoft/layoutlmv3-base` for token classification using the Phase 1 word-level OCR, normalized boxes, and BIO tags. It uses training and validation only; the official test split remains untouched for Phase 4 evaluation.

The notebook stores the best model and processor on Google Drive, logs validation loss and entity-level F1, saves resumable training state each epoch, and reloads the best checkpoint for a final sanity check.

LayoutLMv3 supports token classification with image, words, boxes, and word labels; its processor can disable built-in OCR because this project provides annotated words and boxes. [Hugging Face LayoutLMv3 documentation](https://huggingface.co/docs/transformers/main/model_doc/layoutlmv3).

## Before you start

1. In Colab, choose **Runtime → Change runtime type → GPU** if a GPU is available. Colab does not guarantee a specific accelerator.
2. Make sure Phase 1 prepared `train.jsonl`, `validation.jsonl`, and images on Drive.
3. Run this notebook top-to-bottom. Do not use the test split for model selection.
4. The notebook installs Transformers and metric packages but keeps Colab's preinstalled CUDA-enabled PyTorch.

If there is no GPU, the notebook stops before full training. Set `RUN_CPU_SMOKE_TEST = True` in Cell 1 only if you want a very small CPU smoke run (8 train documents, 4 validation documents, one epoch). That is not full fine-tuning.

## Cell 1 — Check runtime, show GPU, install training dependencies

In [ ]:
import sys, platform, torch
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
GPU_AVAILABLE = torch.cuda.is_available()
if GPU_AVAILABLE:
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GiB):", round(props.total_memory / 1024**3, 2))
else:
    print("GPU unavailable.")

RUN_CPU_SMOKE_TEST = False
if not GPU_AVAILABLE and not RUN_CPU_SMOKE_TEST:
    raise RuntimeError(
        "Full LayoutLMv3 fine-tuning is configured for a Colab GPU. "
        "To run only a small CPU smoke test, set RUN_CPU_SMOKE_TEST = True and rerun this cell."
    )

%pip -q install --upgrade "transformers==5.19.0" "huggingface-hub>=1.31,<2.0" safetensors

## Cell 2 — Mount Drive and define project paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, os, random, shutil, math, time
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

DRIVE_ROOT = Path("/content/drive/MyDrive/intelligent-document-ai")
DATA_PROCESSED = DRIVE_ROOT / "data" / "processed"
ANNOTATIONS = DRIVE_ROOT / "data" / "annotations"
if RUN_CPU_SMOKE_TEST:
    MODEL_DIR = Path("/content/layoutlmv3-sroie-cpu-smoke")
    CHECKPOINT_DIR = Path("/content/layoutlmv3-cpu-smoke-checkpoints")
else:
    MODEL_DIR = DRIVE_ROOT / "models" / "layoutlmv3-sroie-best"
    CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "layoutlmv3-sroie"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
STATE_PATH = CHECKPOINT_DIR / "resume_state.pt"
HISTORY_PATH = CHECKPOINT_DIR / "training_history.json"
BEST_METRICS_PATH = CHECKPOINT_DIR / "best_validation_metrics.json"
print("Model directory:", MODEL_DIR)
print("Resume state:", STATE_PATH)

## Cell 3 — Load and verify prepared train/validation data
This reads the persistent JSONL outputs from Phase 1. It intentionally does not load the test split. Images are copied to Colab's temporary disk to reduce repeated Drive reads during epochs.

In [ ]:
def read_jsonl(path):
    rows = []
    with Path(path).open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON at {path}:{line_number}") from exc
    return rows

metadata_path = ANNOTATIONS / "phase1_metadata.json"
if not metadata_path.is_file():
    raise FileNotFoundError("Missing Phase 1 metadata. Complete the Phase 1 notebook first.")
phase1_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
LABELS = phase1_metadata["label_names"]
if not LABELS or LABELS[0] != "O":
    raise ValueError(f"Unexpected BIO label mapping in Phase 1 metadata: {LABELS}")
LABEL2ID = {label: index for index, label in enumerate(LABELS)}
ID2LABEL = {index: label for label, index in LABEL2ID.items()}

train_records = read_jsonl(DATA_PROCESSED / "train.jsonl")
val_records = read_jsonl(DATA_PROCESSED / "validation.jsonl")
if not train_records or not val_records:
    raise ValueError("Training and validation JSONL files must both contain documents.")

for split_name, rows in (("train", train_records), ("validation", val_records)):
    for row in rows:
        if not (len(row["words"]) == len(row["bboxes"]) == len(row["ner_tags"])):
            raise ValueError(f"Word/box/tag mismatch in {split_name} document {row['document_id']}")
        if any(tag not in LABEL2ID for tag in row["ner_tags"]):
            raise ValueError(f"Unknown BIO label in {split_name} document {row['document_id']}")
        if not Path(row["image_path"]).is_file():
            raise FileNotFoundError(f"Missing image: {row['image_path']}")

train_ids = {str(row["document_id"]) for row in train_records}
val_ids = {str(row["document_id"]) for row in val_records}
if train_ids & val_ids:
    raise ValueError("Document leakage: train and validation IDs overlap.")
print("BIO labels:", LABELS)
print("Training documents:", len(train_records), "| validation documents:", len(val_records))

## Cell 4 — Set seeds and make local image copies
The split is already fixed in Phase 1. These seeds make the training order and model initialization reproducible where hardware kernels allow it.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

LOCAL_IMAGE_DIR = Path("/content/idai_phase3_images")
LOCAL_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

def localize_images(rows, split_name):
    for row in tqdm(rows, desc=f"Copy {split_name} images to local runtime"):
        source = Path(row["image_path"])
        target = LOCAL_IMAGE_DIR / split_name / source.name
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.is_file():
            shutil.copy2(source, target)
        row["image_path"] = str(target)

localize_images(train_records, "train")
localize_images(val_records, "validation")
print("Training images are cached under:", LOCAL_IMAGE_DIR)

## Cell 5 — Load LayoutLMv3 processor and initialize the token classifier
`apply_ocr=False` is important: SROIE already supplies words and boxes. The processor applies image transforms and tokenizes words; `word_labels` lets the LayoutLMv3 tokenizer align labels to subwords and use `-100` for special tokens and non-first subwords. The model's maximum sequence length is bounded at 512.

In [ ]:
from transformers import AutoProcessor, LayoutLMv3ForTokenClassification

MODEL_NAME = "microsoft/layoutlmv3-base"
MAX_LENGTH = 512
processor = AutoProcessor.from_pretrained(MODEL_NAME, apply_ocr=False)
model = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
print("Processor:", type(processor).__name__)
print("Model labels:", model.config.num_labels)
print("Model maximum positions:", model.config.max_position_embeddings)

## Cell 6 — Create a PyTorch Dataset and inspect tensor shapes
One dataset item is one receipt. Its tensors include input IDs, attention mask, bounding boxes, image pixels, and aligned token labels.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class ReceiptTokenDataset(Dataset):
    def __init__(self, records, processor, label2id, max_length=512):
        self.records = records
        self.processor = processor
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        image = Image.open(row["image_path"]).convert("RGB")
        label_ids = [self.label2id[tag] for tag in row["ner_tags"]]
        encoded = self.processor(
            images=image,
            text=row["words"],
            boxes=row["bboxes"],
            word_labels=label_ids,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        # Remove the single-document batch dimension.
        return {key: value.squeeze(0) for key, value in encoded.items()}

train_dataset = ReceiptTokenDataset(train_records, processor, LABEL2ID, MAX_LENGTH)
val_dataset = ReceiptTokenDataset(val_records, processor, LABEL2ID, MAX_LENGTH)

SMOKE_BATCH_SIZE = 1 if RUN_CPU_SMOKE_TEST else 2
train_loader = DataLoader(train_dataset, batch_size=SMOKE_BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=SMOKE_BATCH_SIZE, shuffle=False, num_workers=0)

example_batch = next(iter(train_loader))
for key, value in example_batch.items():
    print(key, tuple(value.shape), value.dtype)
assert example_batch["input_ids"].shape[1] == MAX_LENGTH
assert example_batch["labels"].shape == example_batch["input_ids"].shape
assert example_batch["pixel_values"].shape[1] == 3

## Cell 7 — Configure training, early stopping, and resume
Default settings follow the project plan. If a GPU runs out of memory, reduce `BATCH_SIZE` to 1 and increase `GRAD_ACCUM_STEPS`. Each completed epoch writes resumable optimizer/model state to Drive and the best validation-F1 model to the models folder. Saving optimizer state uses additional Drive space.

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

if RUN_CPU_SMOKE_TEST:
    train_records = train_records[:min(8, len(train_records))]
    val_records = val_records[:min(4, len(val_records))]
    # Recreate datasets/loaders over the intentionally tiny smoke-test subsets.
    train_dataset = ReceiptTokenDataset(train_records, processor, LABEL2ID, max_length=256)
    val_dataset = ReceiptTokenDataset(val_records, processor, LABEL2ID, max_length=256)
    MAX_LENGTH = 256
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)
    EPOCHS, BATCH_SIZE, GRAD_ACCUM_STEPS = 1, 1, 1
else:
    EPOCHS = 10
    BATCH_SIZE = 2
    GRAD_ACCUM_STEPS = 2

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 3
WARMUP_RATIO = 0.1
USE_AMP = torch.cuda.is_available() and not RUN_CPU_SMOKE_TEST
MAX_GRAD_NORM = 1.0
DEVICE = torch.device("cpu" if RUN_CPU_SMOKE_TEST else ("cuda" if torch.cuda.is_available() else "cpu"))
model.to(DEVICE)

# Rebuild loaders using the selected batch size (GPU default is 2).
if not RUN_CPU_SMOKE_TEST:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
optimizer_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_optimizer_steps = optimizer_steps_per_epoch * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(total_optimizer_steps * WARMUP_RATIO),
    num_training_steps=total_optimizer_steps,
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

CONFIG = {
    "model_name": MODEL_NAME, "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS, "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE, "seed": SEED,
    "labels": LABELS, "amp": USE_AMP, "cpu_smoke_test": RUN_CPU_SMOKE_TEST,
}
print("Device:", DEVICE, "| AMP:", USE_AMP, "| optimizer steps:", total_optimizer_steps)
print(json.dumps(CONFIG, indent=2))

## Cell 8 — Training and validation functions
Validation F1 is entity-level BIO span F1 computed directly from tag sequences; subword and special-token positions marked `-100` are ignored. Validation loss is averaged over documents.

In [ ]:
from contextlib import nullcontext

def autocast_context():
    if USE_AMP:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()

def labels_to_sequences(logits, label_ids):
    predicted_ids = logits.argmax(dim=-1).detach().cpu().tolist()
    gold_ids = label_ids.detach().cpu().tolist()
    predictions, references = [], []
    for predicted, gold in zip(predicted_ids, gold_ids):
        doc_pred, doc_gold = [], []
        for pred_id, gold_id in zip(predicted, gold):
            if gold_id == -100:
                continue
            doc_pred.append(ID2LABEL[int(pred_id)])
            doc_gold.append(ID2LABEL[int(gold_id)])
        predictions.append(doc_pred)
        references.append(doc_gold)
    return predictions, references

def evaluate_model(model, loader):
    model.eval()
    loss_total, document_count = 0.0, 0
    all_predictions, all_references = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validation", leave=False):
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            with autocast_context():
                outputs = model(**batch)
            batch_size = batch["input_ids"].shape[0]
            loss_total += float(outputs.loss.detach().float().cpu()) * batch_size
            document_count += batch_size
            pred, gold = labels_to_sequences(outputs.logits, batch["labels"])
            all_predictions.extend(pred)
            all_references.extend(gold)
    def spans_for_document(tags, document_index):
        spans = set()
        active_label, active_start = None, None
        def close(end_index):
            if active_label is not None:
                spans.add((document_index, active_start, end_index, active_label))
        for position, tag in enumerate(tags):
            if tag == "O":
                close(position)
                active_label, active_start = None, None
            elif tag.startswith("B-"):
                close(position)
                active_label, active_start = tag[2:], position
            elif tag.startswith("I-") and active_label == tag[2:]:
                continue
            elif tag.startswith("I-"):
                # Treat an orphan or mismatched I-tag as a new span.
                close(position)
                active_label, active_start = tag[2:], position
            else:
                close(position)
                active_label, active_start = None, None
        close(len(tags))
        return spans

    predicted_spans, reference_spans = set(), set()
    for doc_index, (pred_tags, gold_tags) in enumerate(zip(all_predictions, all_references)):
        predicted_spans.update(spans_for_document(pred_tags, doc_index))
        reference_spans.update(spans_for_document(gold_tags, doc_index))
    true_positive = len(predicted_spans & reference_spans)
    false_positive = len(predicted_spans - reference_spans)
    false_negative = len(reference_spans - predicted_spans)
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
    entity_f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "val_loss": loss_total / max(document_count, 1),
        "precision": precision,
        "recall": recall,
        "entity_f1": entity_f1,
    }

def save_training_state(epoch, best_f1, epochs_without_improvement, history):
    state = {
        "epoch": epoch,
        "best_f1": best_f1,
        "epochs_without_improvement": epochs_without_improvement,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "history": history,
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "config": CONFIG,
    }
    local_state = Path("/content/layoutlmv3_resume_state.pt")
    temporary_drive_state = STATE_PATH.with_suffix(".pt.tmp")
    torch.save(state, local_state)
    shutil.copy2(local_state, temporary_drive_state)
    os.replace(temporary_drive_state, STATE_PATH)

## Cell 9 — Resume if possible, then fine-tune
If a saved resume state exists, this restores the current epoch's model and optimizer state. A separate `layoutlmv3-sroie-best` directory holds the best validation-F1 model for inference. Set `START_FRESH = True` to intentionally ignore an old resume state.

In [ ]:
START_FRESH = False
start_epoch = 0
best_f1 = -1.0
epochs_without_improvement = 0
history = []

if STATE_PATH.is_file() and not START_FRESH and not RUN_CPU_SMOKE_TEST:
    print("Resuming from:", STATE_PATH)
    resume = torch.load(STATE_PATH, map_location="cpu", weights_only=False)
    if resume["config"]["labels"] != CONFIG["labels"]:
        raise ValueError("Saved checkpoint label mapping differs from the current Phase 1 mapping.")
    model.load_state_dict(resume["model_state_dict"])
    optimizer.load_state_dict(resume["optimizer_state_dict"])
    scheduler.load_state_dict(resume["scheduler_state_dict"])
    scaler.load_state_dict(resume["scaler_state_dict"])
    start_epoch = int(resume["epoch"])
    best_f1 = float(resume["best_f1"])
    epochs_without_improvement = int(resume["epochs_without_improvement"])
    history = resume["history"]
    random.setstate(resume["python_rng_state"])
    np.random.set_state(resume["numpy_rng_state"])
    torch.set_rng_state(resume["torch_rng_state"])
    if torch.cuda.is_available() and resume["cuda_rng_state"] is not None:
        torch.cuda.set_rng_state_all(resume["cuda_rng_state"])
elif STATE_PATH.exists() and START_FRESH:
    print("Ignoring the existing resume state. The old file is left on Drive.")

if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
    print("Saved state already met the early-stopping limit; no further epochs will run.")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    running_loss, seen_documents = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    progress = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch + 1}/{EPOCHS}")
    for step, batch in progress:
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        # Correctly average the final, possibly smaller, gradient-accumulation group.
        group_start = (step // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
        group_size = min(GRAD_ACCUM_STEPS, len(train_loader) - group_start)
        try:
            with autocast_context():
                outputs = model(**batch)
                loss = outputs.loss
            scaler.scale(loss / group_size).backward()
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                raise RuntimeError(
                    "GPU out of memory. Set BATCH_SIZE = 1, increase GRAD_ACCUM_STEPS, "
                    "and rerun from Cell 7. If needed, reduce MAX_LENGTH to 256."
                ) from exc
            raise

        batch_size = batch["input_ids"].shape[0]
        running_loss += float(loss.detach().float().cpu()) * batch_size
        seen_documents += batch_size
        is_update_step = ((step + 1) % GRAD_ACCUM_STEPS == 0) or (step + 1 == len(train_loader))
        if is_update_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        progress.set_postfix(loss=f"{running_loss / max(seen_documents, 1):.4f}")

    train_loss = running_loss / max(seen_documents, 1)
    validation = evaluate_model(model, val_loader)
    row = {"epoch": epoch + 1, "train_loss": train_loss, **validation}
    history.append(row)
    print(row)

    improved = validation["entity_f1"] > best_f1
    if improved:
        best_f1 = validation["entity_f1"]
        epochs_without_improvement = 0
        if not RUN_CPU_SMOKE_TEST:
            model.save_pretrained(MODEL_DIR, safe_serialization=True)
            processor.save_pretrained(MODEL_DIR)
            (MODEL_DIR / "label_mapping.json").write_text(
                json.dumps({"label2id": LABEL2ID, "id2label": ID2LABEL}, indent=2),
                encoding="utf-8",
            )
            (MODEL_DIR / "training_config.json").write_text(
                json.dumps(CONFIG, indent=2), encoding="utf-8"
            )
            (BEST_METRICS_PATH).write_text(json.dumps(row, indent=2), encoding="utf-8")
        else:
            model.save_pretrained(MODEL_DIR, safe_serialization=True)
            processor.save_pretrained(MODEL_DIR)
        print("Saved new best checkpoint.")
    else:
        epochs_without_improvement += 1

    if not RUN_CPU_SMOKE_TEST:
        save_training_state(epoch + 1, best_f1, epochs_without_improvement, history)
        HISTORY_PATH.write_text(json.dumps(history, indent=2), encoding="utf-8")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping: validation entity F1 has not improved.")
        break

if not history:
    print("No new epochs ran. The checkpoint may already have completed the configured epoch count.")

## Cell 10 — Review training history and reload the best checkpoint
The held-out test set is not evaluated here; Phase 4 handles that once model selection is finished.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

if history:
    history_df = pd.DataFrame(history)
    display(history_df)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="validation")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[1].plot(history_df["epoch"], history_df["entity_f1"], marker="o")
    axes[1].set_title("Validation entity-level F1")
    axes[1].set_xlabel("Epoch")
    plt.tight_layout()
    plt.show()

if MODEL_DIR.exists() and not RUN_CPU_SMOKE_TEST:
    from transformers import LayoutLMv3ForTokenClassification
    reloaded_processor = AutoProcessor.from_pretrained(MODEL_DIR, apply_ocr=False)
    reloaded_model = LayoutLMv3ForTokenClassification.from_pretrained(MODEL_DIR)
    print("Reloaded best model labels:", reloaded_model.config.id2label)
    print("Reloaded processor:", type(reloaded_processor).__name__)
    print("Best model directory:", MODEL_DIR)
elif RUN_CPU_SMOKE_TEST:
    print("CPU smoke checkpoint saved only under /content; it is not the persistent trained model.")

## Checkpoint recovery and expected Drive outputs

The best inference bundle is `models/layoutlmv3-sroie-best/` and contains model weights/config, processor files, label mapping, and training configuration. The checkpoint folder contains `resume_state.pt`, training history, and best validation metrics.

To resume after a Colab disconnect, remount Drive and rerun the notebook with `START_FRESH = False`. It restores the latest completed epoch. To deliberately restart, set `START_FRESH = True`; existing checkpoints remain on Drive.

A GPU out-of-memory error is recoverable: set batch size to 1 and increase gradient accumulation in Cell 7, then rerun setup and resume. If the GPU is unavailable, full CPU training is intentionally stopped; CPU smoke mode trains only a tiny subset and is not a substitute for the full run.

Phase 3 is complete after a full GPU run saves the best model bundle, processor, label mapping, training configuration, validation history, and successfully reloads the saved model. Do not use validation F1 as test performance.